In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd

In [3]:
data_dir = str(PROJECT_ROOT / "data" / "processed" / "TEP") + "/"
references_dir = str(PROJECT_ROOT / "references" / "TEP") + "/"
models_dir = str(PROJECT_ROOT / "models" / "TEP") + "/"
max_lag_selection_dir = models_dir + "max_lag_selection/"
weights_dir = models_dir + "max_lag_selection/weights/"
y_faultfree_dir = models_dir + "max_lag_selection/y/"
y_pred_faultfree_dir = models_dir + "max_lag_selection/y_pred/"
x_faulty_dir = models_dir + "moving_average_window_selection/X/"
y_faulty_dir = models_dir + "moving_average_window_selection/y/"
y_pred_faulty_dir = models_dir + "moving_average_window_selection/y_pred/"

In [4]:
import os

for directory in [models_dir, max_lag_selection_dir, weights_dir, y_faultfree_dir, y_pred_faultfree_dir, x_faulty_dir, y_faulty_dir, y_pred_faulty_dir]:
    os.makedirs(directory, exist_ok=True)

In [5]:
test_file_name = "TEP_Testing.parquet"
adjacency_matrix_file_name = "TEGroundTruth.txt"

In [6]:
df = pd.read_parquet(data_dir + test_file_name)

In [7]:
n_faulty_samples = df[df["faultNumber"] != 0].shape[0]
df_faulty = df.iloc[-n_faulty_samples:]

n_samples_per_faulty_ts = df[df["faultNumber"] != 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faulty_ts = df[df["faultNumber"] != 0].groupby(["sample"]).size().max()

df_faulty = df_faulty.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [8]:
adjacency_matrix = np.loadtxt(references_dir + adjacency_matrix_file_name)

In [9]:
feature_names = df_faulty.columns
n_features = len(feature_names)
child_parents_pairs = {}

for child_idx in range(n_features):
    parents = []
    for feature_idx in range(n_features):
        if adjacency_matrix[feature_idx, child_idx] == 1:
            parents.append(feature_names[feature_idx])
    child_parents_pairs[feature_names[child_idx]] = parents

In [10]:
process_children = [child for child in child_parents_pairs.keys() if len(child_parents_pairs[child]) != 0 and int(child[1:]) < 23]
n_process_children = len(process_children)
process_child_parents_pairs = {child: child_parents_pairs[child] for child in process_children}

In [11]:
max_lag = 1
sample_precision = np.load(max_lag_selection_dir + "sample_precision_train.npy")

In [12]:
n_samples_test = n_faulty_ts * (n_samples_per_faulty_ts - max_lag)
res_test = np.empty((n_samples_test, n_process_children))

for child, parents in process_child_parents_pairs.items():

    df_X_test = df_faulty[parents + [child]]
    y_test = df_faulty[child]

    df_X_test_lagged = pd.concat([df_X_test.shift(i).add_suffix(f"_lag{i}") for i in range(1, max_lag + 1)], axis=1)

    mask = []
    for ts_start_idx in range(0, n_faulty_samples, n_samples_per_faulty_ts):
        mask.extend(range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_faulty_ts))

    df_X_test_lagged = df_X_test_lagged.iloc[mask]
    y_test = y_test.iloc[mask]

    X_test = np.ones((df_X_test_lagged.shape[0], df_X_test_lagged.shape[1] + 1))
    X_test[:, 1:] = df_X_test_lagged.values
    y_test = y_test.to_numpy()

    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")

    y_test_pred = X_test @ beta

    res_test[:, process_children.index(child)] = y_test - y_test_pred   

In [13]:
scores_raw = np.mean(res_test @ sample_precision * res_test, axis=1)
scores_2d = scores_raw.reshape((n_faulty_ts, n_samples_per_faulty_ts - max_lag))
df_scores_2d = pd.DataFrame(scores_2d)

In [14]:
n_faultfree_samples_per_ts = 160
faultfree_idxs = range(n_faultfree_samples_per_ts - max_lag)

y_anomaly_2d = np.ones_like(scores_2d, dtype=int)
y_anomaly_2d[:, faultfree_idxs] = 0

In [15]:
window = 6
consecutive_alarms = 5
threshold = 1.7085420289572733

y_anomaly_2d_red = y_anomaly_2d[:, window-1:]
idx_first_anomaly = n_faultfree_samples_per_ts - max_lag - window + 1

y_ts_anomaly_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_2d[:, 0] = y_anomaly_2d_red[:, :idx_first_anomaly].max(axis=1)
y_ts_anomaly_2d[:, 1] = y_anomaly_2d_red[:, idx_first_anomaly:].max(axis=1)
y_ts_anomaly = y_ts_anomaly_2d.reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample Accuracy",
    "Sample TPR",
    "Sample FPR",
    "Sample F1",
    "Trajectory-Phase Accuracy",
    "Trajectory-Phase TPR",
    "Trajectory-Phase FPR",
    "Trajectory-Phase F1",
    "Median Anomaly Detection Delay"
])

scores_2d_smoothed = df_scores_2d.T.rolling(window=window).mean().values.T
scores_2d_smoothed = scores_2d_smoothed[:, window-1:]

raw_breaches = (scores_2d_smoothed > threshold).astype(int)

y_anomaly_pred_2d = (
    pd.DataFrame(raw_breaches.T).rolling(
        window=consecutive_alarms, min_periods=consecutive_alarms
    ).sum() == consecutive_alarms
).astype(int).values.T

eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
eval_mask_2d[:, :consecutive_alarms-1] = False

y_anomaly = y_anomaly_2d_red.reshape(-1)
y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
eval_mask = eval_mask_2d.reshape(-1)

y_anomaly_eval = y_anomaly[eval_mask]
y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

tp = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 1))
fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
fn = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 0))
tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

accuracy = (tp + tn) / (tp + fp + fn + tn)
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

y_ts_anomaly_pred_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_pred_2d[:, 0] = y_anomaly_pred_2d[:, :idx_first_anomaly].max(axis=1)

faulty_phase_preds = y_anomaly_pred_2d[:, idx_first_anomaly:]
y_ts_anomaly_pred_2d[:, 1] = faulty_phase_preds.max(axis=1)

detected_mask = faulty_phase_preds.any(axis=1)
if np.any(detected_mask):
    idx_first_anomaly_pred = faulty_phase_preds[detected_mask].argmax(axis=1)
    median_idx_first_anomaly_pred = np.median(idx_first_anomaly_pred)
else:
    median_idx_first_anomaly_pred = np.nan

y_ts_anomaly_pred = y_ts_anomaly_pred_2d.reshape(-1)

tp_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 1))
fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
fn_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 0))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

accuracy_ts = (tp_ts + tn_ts) / (tp_ts + fp_ts + fn_ts + tn_ts)
tpr_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0
precision_ts = tp_ts / (tp_ts + fp_ts) if (tp_ts + fp_ts) > 0 else 0
recall_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
f1_ts = 2 * precision_ts * recall_ts / (precision_ts + recall_ts) if precision_ts + recall_ts > 0 else 0

df_result = pd.DataFrame({
    "Window Size": [window],
    "Consecutive Alarms": [consecutive_alarms],
    "Threshold": [threshold],
    "Sample Accuracy": [accuracy],
    "Sample TPR": [tpr],
    "Sample FPR": [fpr],
    "Sample F1": [f1],
    "Trajectory-Phase Accuracy": [accuracy_ts],
    "Trajectory-Phase TPR": [tpr_ts],
    "Trajectory-Phase FPR": [fpr_ts],
    "Trajectory-Phase F1": [f1_ts],
    "Median Anomaly Detection Delay": [median_idx_first_anomaly_pred],
})

df_results = pd.concat([df_results, df_result], ignore_index=True)

with pd.option_context('display.max_rows', None):
    display(df_results)

<ipykernel>\1473891158.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
0,6,5,1.708542,0.581203,0.502726,0.000253,0.669064,0.92645,0.8709,0.018,0.922124,16.0


In [16]:
from carl.tep_figure_exports import save_tep_figure_metrics
save_tep_figure_metrics(df_results, PROJECT_ROOT, 'Graph-informed', 'faulty_test', 'main_detection')


Saved TEP figure metrics to <repo>\reports\results\TEP\figure_metrics\main_detection_graph_informed_faulty_test.csv


In [17]:
from carl.tep_figure_exports import save_tep_figure_metrics
save_tep_figure_metrics(df_results, PROJECT_ROOT, 'Complete detector', 'faulty_test', 'alarm_ablations')


Saved TEP figure metrics to <repo>\reports\results\TEP\figure_metrics\alarm_ablations_complete_detector_faulty_test.csv
